In [90]:
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [91]:
df = pd.read_csv("/Users/raghav/Downloads/Company Names.csv")

In [92]:
df.head()

,Company Name,Rating,Review Count,Company Type,Headquarters,Company Age,Number of Employees
0,TCS,3.9,16.1k Reviews,Public,Mumbai + 156 more,52 years old,10000+ employees
1,Accenture,4.0,13.9k Reviews,Private,Dublin + 40 more,31 years old,10000+ employees
2,ICICI Bank,4.1,12.6k Reviews,Public,Mumbai + 42 more,26 years old,10000+ employees
3,Cognizant,3.9,12k Reviews,Private,Teaneck + 39 more,26 years old,10000+ employees
4,HDFC Bank,4.0,10.8k Reviews,Public,Mumbai + 46 more,26 years old,10000+ employees


In [93]:
names = list(df["Company Name"])
names[:5]

['TCS', 'Accenture', 'ICICI Bank', 'Cognizant', 'HDFC Bank']

In [94]:
names = [name.lower() for name in names]
names = [''.join(c for c in name if c.isalpha() or c == ' ') for name in names]
names = [name.strip() for name in names]
names = [name for name in names if name]

In [95]:
chars = sorted(list(set(''.join(names))))
chars

[' ',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z',
 'ä',
 'é',
 'í',
 'ü',
 'ā',
 'š']

In [96]:
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['~'] = 0
stoi 

{' ': 1,
 'a': 2,
 'b': 3,
 'c': 4,
 'd': 5,
 'e': 6,
 'f': 7,
 'g': 8,
 'h': 9,
 'i': 10,
 'j': 11,
 'k': 12,
 'l': 13,
 'm': 14,
 'n': 15,
 'o': 16,
 'p': 17,
 'q': 18,
 'r': 19,
 's': 20,
 't': 21,
 'u': 22,
 'v': 23,
 'w': 24,
 'x': 25,
 'y': 26,
 'z': 27,
 'ä': 28,
 'é': 29,
 'í': 30,
 'ü': 31,
 'ā': 32,
 'š': 33,
 '~': 0}

In [97]:
itos = {i:s for s,i in stoi.items()}
itos

{1: ' ',
 2: 'a',
 3: 'b',
 4: 'c',
 5: 'd',
 6: 'e',
 7: 'f',
 8: 'g',
 9: 'h',
 10: 'i',
 11: 'j',
 12: 'k',
 13: 'l',
 14: 'm',
 15: 'n',
 16: 'o',
 17: 'p',
 18: 'q',
 19: 'r',
 20: 's',
 21: 't',
 22: 'u',
 23: 'v',
 24: 'w',
 25: 'x',
 26: 'y',
 27: 'z',
 28: 'ä',
 29: 'é',
 30: 'í',
 31: 'ü',
 32: 'ā',
 33: 'š',
 0: '~'}

In [98]:
vocab_size = len(stoi)
vocab_size

34

In [99]:
context_size = 8
def build_dataset(names):
    X = []
    Y = []
    for name in names:
        cs = [0] * context_size
        for ch in name + '~':
            ix = stoi[ch]
            X.append(cs)
            Y.append(ix)
            cs = cs[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)

    return X,Y

In [100]:
import random
random.seed(42)
random.shuffle(names)
n1 = int(0.8 * len(names))
n2 = int(0.9 * len(names))

Xtr,Ytr = build_dataset(names[:n1])
Xdev,Ydev = build_dataset(names[n1:n2])
Xte,Yte = build_dataset(names[n2:])

In [101]:
print(Xtr.shape,Ytr.shape)

torch.Size([138602, 8]) torch.Size([138602])


In [102]:
n_embd = 10
C = torch.randn(vocab_size,10)
C.shape

torch.Size([34, 10])

In [103]:
ix = torch.randint(0,Xtr.shape[0],(4,))
embd = C[Xtr[ix]]

In [104]:
embd.shape

torch.Size([4, 8, 10])

In [105]:
class Linear:
    def __init__(self,fan_in,fan_out,bias=True):
        self.weights = torch.randn(fan_in,fan_out) / fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self,x):
        self.out = x @ self.weights
        if self.bias is not None:
            self.out += self.bias
        return self.out 

    def parameters(self):
        return [self.weights] + ([] if self.bias is None else [self.bias])


class BatchNorm1d:
    def __init__(self,n_features,eps = 1e-5,momentum=0.1):
        self.eps = eps
        self.momentum = momentum
        self.gamma = torch.ones(n_features)
        self.beta = torch.zeros(n_features)

        self.running_mean = torch.zeros(n_features)
        self.running_var = torch.ones(n_features)
        self.training = True

    def __call__(self,x):
        if self.training:
            if x.ndim == 2:
                dim = 0
            elif x.ndim ==3:
                dim = (0,1)
            xmean = x.mean(dim,keepdim=True)
            xvar = x.var(dim,keepdim=True)
        else:
            xmean = self.running_mean
            xvar = self.running_var

        xhat = (x-xmean)/ torch.sqrt(xvar+self.eps)
        self.out = self.gamma * xhat + self.beta
        
        if self.training:
            with torch.no_grad():
                self.running_mean = (1 - self.momentum) * self.running_mean + self.momentum * xmean
                self.running_var = (1 - self.momentum) * self.running_var + self.momentum * xvar

        return self.out

    def parameters(self):
        return [self.gamma,self.beta]

class Tanh:
  def __call__(self, x):
    self.out = torch.tanh(x)
    return self.out
  def parameters(self):
    return []


class Embedding:
    def __init__(self,num_embeddings,embeddings_dim):
        self.weights = torch.randn(num_embeddings,embeddings_dim)

    def __call__(self,IX):
        self.out = self.weights[IX]
        return self.out

    def parameters(self):
        return [self.weights]

class FlattenConsecutive:
    def __init__(self,n):
        self.n = n

    def __call__(self,x):
        B,T,C = x.shape
        x = x.view(B,T//self.n,C*self.n)
        if x.shape[1] == 1:
           x = x.squeeze(1)

        self.out = x
        return self.out

    def parameters(self):
        return []

class Sequential:

    def __init__(self,layers):
        self.layers = layers

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)

        self.out = x
        return self.out

    def parameters(self):

        return [p for layer in self.layers for p in layer.parameters()]

In [106]:
n_embd = 10 
n_hidden = 200
vocab_size = len(stoi)

In [107]:
model = Sequential([
    Embedding(vocab_size,n_embd),
    FlattenConsecutive(2), Linear(2*n_embd,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    FlattenConsecutive(2), Linear(2*n_hidden,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    FlattenConsecutive(2), Linear(2*n_hidden,n_hidden,bias=False),BatchNorm1d(n_hidden),Tanh(),
    Linear(n_hidden,vocab_size)
])

In [108]:
parameters = model.parameters()
print(sum(p.nelement() for p in parameters))
for p in parameters:
  p.requires_grad = True

172374


In [109]:
max_steps = 200000
batch_size = 32


for i in range(max_steps):
    ix = torch.randint(0,Xtr.shape[0],(batch_size,))
    Xb,Yb = Xtr[ix] , Ytr[ix]

    logits = model(Xb)

    loss = F.cross_entropy(logits,Yb)

    for p in parameters:
        p.grad = None

    loss.backward()

    lr = 0.1 if i < 100000 else 0.01

    for p in parameters:
        p.data += -lr * p.grad

    if i % 10000 == 0: # print every once in a while
        print(f'{i:7d}/{max_steps:7d}: {loss.item():.4f}')

      0/ 200000: 3.8720
  10000/ 200000: 1.7512
  20000/ 200000: 1.8526
  30000/ 200000: 1.5792
  40000/ 200000: 1.7965
  50000/ 200000: 1.1065
  60000/ 200000: 1.2886
  70000/ 200000: 1.4207
  80000/ 200000: 1.4613
  90000/ 200000: 0.8586
 100000/ 200000: 1.0867
 110000/ 200000: 1.5278
 120000/ 200000: 1.3421
 130000/ 200000: 0.8462
 140000/ 200000: 1.7322
 150000/ 200000: 1.4849
 160000/ 200000: 1.2631
 170000/ 200000: 1.3453
 180000/ 200000: 1.0103
 190000/ 200000: 1.0470


In [110]:
for layer in model.layers:
    layer.training = False

In [111]:
@torch.no_grad()
def split_loss(split):
    x,y = {
        'train': (Xtr,Ytr),
        'val' : (Xdev,Ydev),
        'test':(Xte,Yte),
    }[split]
    logits = model(x)
    loss = F.cross_entropy(logits,y)
    print(split,loss.item())

split_loss('train')
split_loss('val')

train 1.1391631364822388
val 1.8266135454177856


In [112]:
for _ in range(20):
    out = []
    context = [0] * context_size
    while True:
        x = torch.tensor([context])
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break
    print(''.join(itos[i] for i in out if itos[i] != '~'))

jcc
spilliss
reva biomex industries
ality rougs integrative  manufactu
univpaints digital
axis business solutions
abn nird research print
netto public school
abyan international
innovative healthcare
bajaj  shriytigar
jubilant
lifent service infovaty medical college
wipro infotel
nill
bharat heavy industries
ics cable
campha electroloss
caltics
sala state ex
